# Second Quantization of the Harmonic Oscillator

In the previous notebook we solved the quantum harmonic oscillator (QHO) the "textbook" way: we wrote the Hamiltonian as a differential operator in the position variable $x$, found the wave functions $\psi_v(x)$ as Hermite polynomials times a Gaussian, and read off the energies $E_v = \hbar\omega(v + \tfrac{1}{2})$. That approach, working with wave functions in the **position basis**, is often called *first quantization*.

This notebook develops a second, complementary language for the same physics: **second quantization**, built out of *raising* and *lowering* operators (the "ladder operators"). Two things make this worth your time:

1. **It is often easier.** Once we know how the ladder operators act on the energy eigenstates, many quantities that required integrals over $\psi_v(x)$ become simple bookkeeping.
2. **It is the natural language of the quantized electromagnetic field.** A single mode of light inside a cavity is, mathematically, *another harmonic oscillator*. Everything we build here for molecular vibrations transfers almost symbol-for-symbol to photons. This is the doorway to **cavity quantum electrodynamics (cavity QED)**.

Here is the plan:

- **Part 1: Molecular vibrations.** Recap the ladder operators, then build them as finite matrices in code. We will watch the operators $\hat{x}$, $\hat{p}$, $\hat{x}^2$, $\hat{p}^2$ emerge from the ladder operators, check their matrix elements, and, importantly, see exactly how *truncating* the basis corrupts the commutation relations.
- **Part 2: The quantized cavity field.** Reuse the same machinery for photons. We connect the matter position operator to the **vector potential**, the matter momentum operator to the **electric field**, compute expectation values and fluctuations in photon number states, and set up the composite matter $\otimes$ photon space.

Throughout, the code cells are meant to be run. 

## Part 1: Molecular Vibrations in Second Quantization

### 1.1 Why introduce a new representation?

Recall the one-dimensional harmonic oscillator Hamiltonian for a vibrating molecule (reduced mass $m$, force constant $k = m\omega^2$),

$$
\hat{H} = \frac{1}{2m}\hat{p}^2 + \frac{1}{2}m\omega^2\hat{x}^2 ,
$$

whose energy eigenstates $|n\rangle$ satisfy

$$
\hat{H}\,|n\rangle = \hbar\omega\left(n + \tfrac{1}{2}\right)|n\rangle, \qquad n = 0, 1, 2, \dots
$$

In first quantization, $|n\rangle$ *is* the wave function $\psi_n(x)$, and operators like $\hat{x}$ and $\hat{p}$ act by multiplication and differentiation. To compute something like $\langle n|\hat{x}^2|n\rangle$ we would evaluate an integral $\int \psi_n(x)\, x^2\, \psi_n(x)\, dx$.

Second quantization sidesteps those integrals. The key observation is that the ladder of equally spaced energy levels *suggests* the existence of operators that move us up and down the ladder one rung at a time. If we can express $\hat{x}$ and $\hat{p}$ in terms of those operators, then any expectation value becomes a question of "which rungs connect to which," and the integrals never appear.

### 1.2 The ladder operators

Define the **lowering** (annihilation) operator $\hat{a}$ and the **raising** (creation) operator $\hat{a}^{\dagger}$ through

$$
\hat{a} = \frac{1}{\sqrt{2}}\left(\sqrt{\tfrac{m\omega}{\hbar}}\,\hat{x} + \frac{i}{\sqrt{m\hbar\omega}}\,\hat{p}\right), \qquad
\hat{a}^{\dagger} = \frac{1}{\sqrt{2}}\left(\sqrt{\tfrac{m\omega}{\hbar}}\,\hat{x} - \frac{i}{\sqrt{m\hbar\omega}}\,\hat{p}\right).
$$

The names are earned by how they act on the energy eigenstates:

$$
\hat{a}\,|n\rangle = \sqrt{n}\,|n-1\rangle, \qquad
\hat{a}^{\dagger}\,|n\rangle = \sqrt{n+1}\,|n+1\rangle .
$$

$\hat{a}$ steps *down* the ladder (and annihilates the ground state, $\hat{a}|0\rangle = 0$), while $\hat{a}^{\dagger}$ steps *up*. The $\sqrt{n}$ factors are important: they are exactly what keep the states normalized and they will show up all over the code below!

Two immediate consequences. First, the **number operator**

$$
\hat{N} = \hat{a}^{\dagger}\hat{a}, \qquad \hat{N}\,|n\rangle = n\,|n\rangle
$$

simply counts the quanta (here, vibrational quanta) in the state. Second, the Hamiltonian can be written in an extremely concise way:

$$
\hat{H} = \hbar\omega\left(\hat{a}^{\dagger}\hat{a} + \tfrac{1}{2}\right) = \hbar\omega\left(\hat{N} + \tfrac{1}{2}\right),
$$

from which $E_n = \hbar\omega(n + \tfrac12)$ follows in one line rather than a page of power-series algebra.

<details>
<summary>Click to show how the Hamiltonian collapses to <span>$\hbar\omega(\hat{N}+\tfrac12)$</span></summary>

Start from $\hat{H} = \frac{1}{2m}\hat{p}^2 + \frac{1}{2}m\omega^2\hat{x}^2$ and substitute the inverse relations (derived in the next subsection)

$$
\hat{x} = \sqrt{\tfrac{\hbar}{2m\omega}}\left(\hat{a}^{\dagger} + \hat{a}\right), \qquad
\hat{p} = i\sqrt{\tfrac{m\hbar\omega}{2}}\left(\hat{a}^{\dagger} - \hat{a}\right).
$$

Squaring,

$$
\hat{p}^2 = -\frac{m\hbar\omega}{2}\left(\hat{a}^{\dagger}\hat{a}^{\dagger} - \hat{a}^{\dagger}\hat{a} - \hat{a}\hat{a}^{\dagger} + \hat{a}\hat{a}\right), \qquad
\hat{x}^2 = \frac{\hbar}{2m\omega}\left(\hat{a}^{\dagger}\hat{a}^{\dagger} + \hat{a}^{\dagger}\hat{a} + \hat{a}\hat{a}^{\dagger} + \hat{a}\hat{a}\right).
$$

Insert these into $\hat{H}$. The $\hat{a}^{\dagger}\hat{a}^{\dagger}$ and $\hat{a}\hat{a}$ terms cancel between the kinetic and potential pieces, leaving

$$
\hat{H} = \frac{\hbar\omega}{4}\left(2\hat{a}^{\dagger}\hat{a} + 2\hat{a}\hat{a}^{\dagger}\right) = \frac{\hbar\omega}{2}\left(\hat{a}^{\dagger}\hat{a} + \hat{a}\hat{a}^{\dagger}\right).
$$

Finally use the commutator $\hat{a}\hat{a}^{\dagger} = \hat{a}^{\dagger}\hat{a} + 1$ (next subsection) to get $\hat{H} = \hbar\omega\left(\hat{a}^{\dagger}\hat{a} + \tfrac12\right)$.
</details>

### 1.3 Position, momentum, and the commutators

Inverting the definitions of $\hat{a}$ and $\hat{a}^{\dagger}$ expresses the physical observables back in terms of the ladder operators:

$$
\boxed{\;\hat{x} = \sqrt{\frac{\hbar}{2m\omega}}\left(\hat{a}^{\dagger} + \hat{a}\right), \qquad
\hat{p} = i\sqrt{\frac{m\hbar\omega}{2}}\left(\hat{a}^{\dagger} - \hat{a}\right).\;}
$$

Notice the structural difference that will echo through the whole notebook: **$\hat{x}$ is built from the *sum* $\hat{a}^{\dagger} + \hat{a}$, while $\hat{p}$ is built from the *difference* $\hat{a}^{\dagger} - \hat{a}$** (times $i$). Keep an eye on that sum-vs-difference pattern.

Everything is tied together by a single commutation relation. From the canonical $[\hat{x}, \hat{p}] = i\hbar$ one can show

$$
[\hat{a}, \hat{a}^{\dagger}] = 1,
$$

and, running the logic the other way, the ladder-operator commutator reproduces $[\hat{x}, \hat{p}] = i\hbar$. These two statements are equivalent; each is the fingerprint of quantum mechanics living in this problem. In Part 1 of the code we will build both operators as finite matrices and test these relations directly — and discover that *finite matrices cannot quite satisfy them*, for a reason that turns out to be both deep and practically important.

### 1.4 From operators to matrices

To compute with these operators we choose a basis and write every operator as a matrix. The natural basis is the set of number states $\{|0\rangle, |1\rangle, |2\rangle, \dots\}$ — the "Fock basis." In this basis a state is a column vector,

$$
|0\rangle = \begin{pmatrix} 1 \\ 0 \\ 0 \\ \vdots \end{pmatrix}, \quad
|1\rangle = \begin{pmatrix} 0 \\ 1 \\ 0 \\ \vdots \end{pmatrix}, \quad \dots
$$

and an operator is the matrix whose entries are $\langle m|\hat{O}|n\rangle$.

There is one catch that we cannot avoid and should not hide: the true Fock space is **infinite-dimensional** (there is no largest $n$). A computer can only hold a finite matrix, so we must **truncate** the ladder at some highest state $|N-1\rangle$, keeping an $N \times N$ block. Choosing $N$ well — and understanding what truncation breaks — is a recurring theme in this style of calculation. We start deliberately small ($N=2$), then grow.

In [1]:
import numpy as np

# Print matrices cleanly: 3 decimals, hide tiny -0.000 noise, wide lines
np.set_printoptions(precision=3, suppress=True, linewidth=120)

# Work in a simple unit system so the numbers are easy to read.
# (Restoring hbar, m, omega just rescales x and p by known constants.)
hbar = 1.0
m    = 1.0
omega = 1.0

def annihilation(N):
    """Lowering operator a in the N-state Fock basis {|0>, ..., |N-1>}.

    a|n> = sqrt(n)|n-1>, so the nonzero entries sit on the first
    super-diagonal: <n-1|a|n> = sqrt(n).
    """
    return np.diag(np.sqrt(np.arange(1, N)), k=1)

def creation(N):
    """Raising operator a^dagger = (a)^dagger.  a^dagger|n> = sqrt(n+1)|n+1>."""
    return annihilation(N).conj().T

def basis_state(n, N):
    """Column vector representing the Fock state |n> in an N-dim basis."""
    ket = np.zeros((N, 1))
    ket[n, 0] = 1.0
    return ket

print("Setup complete. hbar = m = omega =", hbar)

Setup complete. hbar = m = omega = 1.0


Below are two small helpers we will reuse constantly: `comm` forms a commutator $[\hat A, \hat B] = \hat A\hat B - \hat B\hat A$, and `expval` computes an expectation value $\langle n|\hat O|n\rangle$ by sandwiching the operator between a Fock ket and its conjugate.

In [2]:
def comm(A, B):
    """Commutator [A, B] = A B - B A."""
    return A @ B - B @ A

def expval(op, n, N):
    """Expectation value <n| op |n> for the Fock state |n> in an N-dim basis."""
    ket = basis_state(n, N)
    return complex((ket.conj().T @ op @ ket).item())

#### Start small: $N = 2$

With only two states, $\{|0\rangle, |1\rangle\}$, the operators are $2\times 2$ matrices. This is the smallest space in which the ladder does anything at all.

In [3]:
N = 2
a  = annihilation(N)
ad = creation(N)

print("Lowering operator a (N=2):")
print(a)
print("\nRaising operator a^dagger (N=2):")
print(ad)

Lowering operator a (N=2):
[[0. 1.]
 [0. 0.]]

Raising operator a^dagger (N=2):
[[0. 0.]
 [1. 0.]]


Read the matrices against the rules $\hat{a}|n\rangle = \sqrt{n}\,|n-1\rangle$ and $\hat{a}^{\dagger}|n\rangle = \sqrt{n+1}\,|n+1\rangle$. The single nonzero entry of $\hat{a}$ (value $\sqrt{1}=1$) sits where it can turn $|1\rangle$ into $|0\rangle$; the single entry of $\hat{a}^{\dagger}$ turns $|0\rangle$ into $|1\rangle$. Let us watch them act.

In [4]:
ket0 = basis_state(0, N)
ket1 = basis_state(1, N)

print("a|0> =", (a @ ket0).ravel(), " -> should be 0 (ground state is annihilated)")
print("a|1> =", (a @ ket1).ravel(), " -> should be sqrt(1)|0> = |0>")
print()
print("a^dag|0> =", (ad @ ket0).ravel(), " -> should be sqrt(1)|1> = |1>")
print("a^dag|1> =", (ad @ ket1).ravel(), " -> should be sqrt(2)|2> ... but |2> is not in our N=2 space!")

a|0> = [0. 0.]  -> should be 0 (ground state is annihilated)
a|1> = [1. 0.]  -> should be sqrt(1)|0> = |0>

a^dag|0> = [0. 1.]  -> should be sqrt(1)|1> = |1>
a^dag|1> = [0. 0.]  -> should be sqrt(2)|2> ... but |2> is not in our N=2 space!


That last line is the whole story of truncation in miniature. The exact rule says $\hat{a}^{\dagger}|1\rangle = \sqrt{2}\,|2\rangle$, but $|2\rangle$ lies *outside* the two-state space we kept, so the finite matrix simply returns zero. The ladder has no top rung in reality; our matrix has forced one. Everything that goes wrong with truncation traces back to this amputated top rung — we will quantify it shortly.

#### A roomier ladder: $N = 4$

Four states make the structure much easier to see. The $\sqrt{n}$ factors ($\sqrt{1}, \sqrt{2}, \sqrt{3}$) now appear explicitly.

In [5]:
N = 4
a  = annihilation(N)
ad = creation(N)

print("Lowering operator a (N=4):")
print(a)
print("\nRaising operator a^dagger (N=4):")
print(ad)

print("\nAction on each basis state (coefficients in the |0>,|1>,|2>,|3> basis):")
for n in range(N):
    ket = basis_state(n, N)
    print(f"  a|{n}>      =", (a  @ ket).ravel(), f"   (expect sqrt({n}) on |{n-1}>)")
for n in range(N):
    ket = basis_state(n, N)
    print(f"  a^dag|{n}>  =", (ad @ ket).ravel(), f"   (expect sqrt({n+1}) on |{n+1}>)")

Lowering operator a (N=4):
[[0.    1.    0.    0.   ]
 [0.    0.    1.414 0.   ]
 [0.    0.    0.    1.732]
 [0.    0.    0.    0.   ]]

Raising operator a^dagger (N=4):
[[0.    0.    0.    0.   ]
 [1.    0.    0.    0.   ]
 [0.    1.414 0.    0.   ]
 [0.    0.    1.732 0.   ]]

Action on each basis state (coefficients in the |0>,|1>,|2>,|3> basis):
  a|0>      = [0. 0. 0. 0.]    (expect sqrt(0) on |-1>)
  a|1>      = [1. 0. 0. 0.]    (expect sqrt(1) on |0>)
  a|2>      = [0.    1.414 0.    0.   ]    (expect sqrt(2) on |1>)
  a|3>      = [0.    0.    1.732 0.   ]    (expect sqrt(3) on |2>)
  a^dag|0>  = [0. 1. 0. 0.]    (expect sqrt(1) on |1>)
  a^dag|1>  = [0.    0.    1.414 0.   ]    (expect sqrt(2) on |2>)
  a^dag|2>  = [0.    0.    0.    1.732]    (expect sqrt(3) on |3>)
  a^dag|3>  = [0. 0. 0. 0.]    (expect sqrt(4) on |4>)


### 1.5 Building $\hat{x}$ and $\hat{p}$

Now assemble the physical observables straight from the boxed formulas of §1.3. Because we set $\hbar = m = \omega = 1$, the prefactors are $\sqrt{\hbar/2m\omega} = 1/\sqrt{2}$ and $\sqrt{m\hbar\omega/2} = 1/\sqrt{2}$.

In [6]:
def position(N):
    """x = sqrt(hbar/2 m omega) (a^dag + a)."""
    return np.sqrt(hbar / (2 * m * omega)) * (creation(N) + annihilation(N))

def momentum(N):
    """p = i sqrt(m hbar omega / 2) (a^dag - a)."""
    return 1j * np.sqrt(m * hbar * omega / 2) * (creation(N) - annihilation(N))

N = 4
x = position(N)
p = momentum(N)

print("x  (real, symmetric — the sum a^dag + a):")
print(x.real)
print("\np  (imaginary, antisymmetric — the difference a^dag - a):")
print(p)   # keep complex so the i is visible

x  (real, symmetric — the sum a^dag + a):
[[0.    0.707 0.    0.   ]
 [0.707 0.    1.    0.   ]
 [0.    1.    0.    1.225]
 [0.    0.    1.225 0.   ]]

p  (imaginary, antisymmetric — the difference a^dag - a):
[[ 0.+0.j    -0.-0.707j  0.+0.j     0.+0.j   ]
 [ 0.+0.707j  0.+0.j    -0.-1.j     0.+0.j   ]
 [ 0.+0.j     0.+1.j     0.+0.j    -0.-1.225j]
 [ 0.+0.j     0.+0.j     0.+1.225j  0.+0.j   ]]


The two matrices display the sum-vs-difference pattern promised earlier. $\hat{x}$ is real and symmetric; $\hat{p}$ is purely imaginary and antisymmetric. Both connect a state $|n\rangle$ only to its immediate neighbors $|n\pm 1\rangle$ — that is why they are called *nearest-neighbor* on the ladder. Acting on a basis state makes this concrete:

In [7]:
ket2 = basis_state(2, N)
print("x|2> =", (x @ ket2).ravel())
print("        -> mixes |1> and |3| with weights sqrt(2)/sqrt(2)=1 and sqrt(3)/sqrt(2)")
print("p|2> =", (p @ ket2).ravel())
print("        -> same neighbors, but with opposite signs and a factor of i")

x|2> = [0.    1.    0.    1.225]
        -> mixes |1> and |3| with weights sqrt(2)/sqrt(2)=1 and sqrt(3)/sqrt(2)
p|2> = [0.+0.j    0.-1.j    0.+0.j    0.+1.225j]
        -> same neighbors, but with opposite signs and a factor of i


### 1.6 Matrix elements of $\hat{x}$, $\hat{x}^2$, $\hat{p}$, $\hat{p}^2$

Expectation values that once required integrals over $\psi_n(x)$ are now matrix diagonals. Two classic results:

- $\langle n|\hat{x}|n\rangle = 0$ and $\langle n|\hat{p}|n\rangle = 0$ — a stationary oscillator has no *average* displacement or momentum (it is equally likely to be found on either side).
- $\langle n|\hat{x}^2|n\rangle = \dfrac{\hbar}{2m\omega}(2n+1)$ and $\langle n|\hat{p}^2|n\rangle = \dfrac{m\hbar\omega}{2}(2n+1)$ — the *spread* grows with $n$.

Let us confirm these against the matrices, and watch the truncation error appear.

In [8]:
N = 4
x = position(N)
p = momentum(N)
x2 = x @ x
p2 = p @ p

print("Diagonal matrix elements <n|O|n> for n = 0..3\n")
print(f"{'n':>2} | {'<x>':>8} {'<p>':>8} | {'<x^2>':>8} {'exact':>8} | {'<p^2>':>8} {'exact':>8}")
print("-" * 66)
for n in range(N):
    ex   = expval(x,  n, N).real
    ep   = expval(p,  n, N).real
    ex2  = expval(x2, n, N).real
    ep2  = expval(p2, n, N).real
    ex2_exact = hbar / (2*m*omega) * (2*n + 1)
    ep2_exact = m*hbar*omega / 2   * (2*n + 1)
    print(f"{n:>2} | {ex:>8.3f} {ep:>8.3f} | {ex2:>8.3f} {ex2_exact:>8.3f} | {ep2:>8.3f} {ep2_exact:>8.3f}")

Diagonal matrix elements <n|O|n> for n = 0..3

 n |      <x>      <p> |    <x^2>    exact |    <p^2>    exact
------------------------------------------------------------------
 0 |    0.000    0.000 |    0.500    0.500 |    0.500    0.500
 1 |    0.000    0.000 |    1.500    1.500 |    1.500    1.500
 2 |    0.000    0.000 |    2.500    2.500 |    2.500    2.500
 3 |    0.000    0.000 |    1.500    3.500 |    1.500    3.500


Look carefully at the top row, $n = 3$. The computed $\langle 3|\hat{x}^2|3\rangle$ and $\langle 3|\hat{p}^2|3\rangle$ **disagree** with the exact values, while every lower state is perfect. The reason is exactly the amputated top rung. The operator $\hat{x}^2$ contains the piece $\hat{a}\hat{a}^{\dagger}$, whose action on $|3\rangle$ is $|3\rangle \xrightarrow{\hat{a}^{\dagger}} \sqrt{4}\,|4\rangle \xrightarrow{\hat{a}} \sqrt{4}\,|3\rangle$. That excursion up to $|4\rangle$ is lost when $|4\rangle$ is not in the basis, so the highest state is undercounted.

This gives a practical rule of thumb worth remembering:

> **To trust the matrix elements of an operator involving up to $k$ ladder operators for states up to $|n_{\max}\rangle$, keep at least $n_{\max} + k$ states in the basis.** For $\hat{x}^2$ or $\hat{p}^2$ ($k=2$), pad your basis by two extra states beyond the highest one you care about.

### 1.7 The number operator and the Hamiltonian

Two more operators fall right out. The number operator $\hat{N} = \hat{a}^{\dagger}\hat{a}$ should be diagonal with entries $0, 1, 2, \dots$, and the Hamiltonian $\hat{H} = \hbar\omega(\hat{N} + \tfrac12)$ should be diagonal with the familiar energies.

In [9]:
N = 4
a  = annihilation(N)
ad = creation(N)

Nop = ad @ a                       # number operator
H   = hbar * omega * (Nop + 0.5 * np.eye(N))

print("Number operator N = a^dag a (diagonal counts the quanta):")
print(Nop.real)
print("\nEigenvalues of H (should be hbar*omega*(n+1/2) = 0.5, 1.5, 2.5, 3.5):")
print(np.linalg.eigvalsh(H))

Number operator N = a^dag a (diagonal counts the quanta):
[[0. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 2. 0.]
 [0. 0. 0. 3.]]

Eigenvalues of H (should be hbar*omega*(n+1/2) = 0.5, 1.5, 2.5, 3.5):
[0.5 1.5 2.5 3.5]


Unlike $\hat{x}^2$ and $\hat{p}^2$, the number operator and Hamiltonian are **exactly right even at the top of the truncated ladder**. That is because $\hat{N} = \hat{a}^{\dagger}\hat{a}$ acts as $|n\rangle \xrightarrow{\hat{a}} \sqrt{n}\,|n-1\rangle \xrightarrow{\hat{a}^{\dagger}} n\,|n\rangle$ — it steps *down first*, so it never needs the missing rung above. The order in which the ladder operators appear determines whether truncation bites.

### 1.8 Commutators and the truncation catch

Now the punchline of Part 1. We claimed $[\hat{a}, \hat{a}^{\dagger}] = 1$ (the identity) and $[\hat{x}, \hat{p}] = i\hbar$. Let us test them on the finite matrices for several basis sizes.

In [10]:
for N in [2, 4, 8]:
    a  = annihilation(N)
    ad = creation(N)
    x  = position(N)
    p  = momentum(N)

    C_ladder = comm(a, ad)      # should be the identity
    C_xp     = comm(x, p)       # should be i*hbar*identity

    print(f"===== N = {N} =====")
    print("diagonal of [a, a^dag]      :", np.diag(C_ladder).real)
    print("diagonal of [x, p] / i      :", (np.diag(C_xp) / 1j).real, " (expect hbar =", hbar, ")")
    print("trace of [a, a^dag]         :", np.trace(C_ladder).real)
    print()

===== N = 2 =====
diagonal of [a, a^dag]      : [ 1. -1.]
diagonal of [x, p] / i      : [ 1. -1.]  (expect hbar = 1.0 )
trace of [a, a^dag]         : 0.0

===== N = 4 =====
diagonal of [a, a^dag]      : [ 1.  1.  1. -3.]
diagonal of [x, p] / i      : [ 1.  1.  1. -3.]  (expect hbar = 1.0 )
trace of [a, a^dag]         : 0.0

===== N = 8 =====
diagonal of [a, a^dag]      : [ 1.  1.  1.  1.  1.  1.  1. -7.]
diagonal of [x, p] / i      : [ 1.  1.  1.  1.  1.  1.  1. -7.]  (expect hbar = 1.0 )
trace of [a, a^dag]         : 0.0



The commutator equals the identity **everywhere except the very last diagonal entry**, which reads $-(N-1)$ instead of $+1$. As $N$ grows, the corruption is pushed higher up the ladder — but it never disappears.

This is not a bug in the code; it is a theorem. For *any* two finite matrices, the trace of a commutator is exactly zero:

$$
\mathrm{Tr}[\hat{A}, \hat{B}] = \mathrm{Tr}(\hat{A}\hat{B}) - \mathrm{Tr}(\hat{B}\hat{A}) = 0,
$$

because the trace is cyclic. But $\mathrm{Tr}(\mathbb{1}) = N \neq 0$. So **no finite matrices can ever satisfy $[\hat{a}, \hat{a}^{\dagger}] = \mathbb{1}$** — the relation genuinely requires the infinite ladder. Our truncated matrices do the next best thing: they get it right on all but the top state, dumping the entire unavoidable discrepancy ($\mathrm{Tr} = 0$ forces the diagonal to sum to zero, so $(N-1)\times(+1)$ must be balanced by one entry of $-(N-1)$) into the last rung.

The practical lesson is the same one from §1.6, now with teeth: **truncation errors live at the top of the basis.** Keep the states you physically care about well below the truncation ceiling, and pad generously. When in doubt, increase $N$ and check that your answer stops changing.

### 1.9 A worked check: the ground state is a minimum-uncertainty state

A capstone for Part 1. The Heisenberg uncertainty principle guarantees $\sigma_x \sigma_p \geq \hbar/2$, and the oscillator ground state is famous for saturating this bound exactly: $\sigma_x \sigma_p = \hbar/2$. Here

$$
\sigma_x = \sqrt{\langle \hat{x}^2\rangle - \langle \hat{x}\rangle^2}, \qquad
\sigma_p = \sqrt{\langle \hat{p}^2\rangle - \langle \hat{p}\rangle^2}.
$$

We can now verify this numerically. Note that even to describe the *ground* state faithfully we need $N \geq 2$, because $\hat{x}^2$ and $\hat{p}^2$ reach up to $|2\rangle$ — a direct application of the padding rule.

In [11]:
def sigma(op, n, N):
    """Standard deviation of observable `op` in Fock state |n>."""
    mean_sq = expval(op @ op, n, N).real
    mean    = expval(op,      n, N).real
    return np.sqrt(mean_sq - mean**2)

N = 6   # padded comfortably above the ground state
x = position(N)
p = momentum(N)

sx = sigma(x, 0, N)
sp = sigma(p, 0, N)
print(f"ground state:  sigma_x = {sx:.4f},  sigma_p = {sp:.4f}")
print(f"product sigma_x * sigma_p = {sx*sp:.4f}")
print(f"hbar/2                    = {hbar/2:.4f}")
print("Minimum-uncertainty state confirmed." if np.isclose(sx*sp, hbar/2) else "Mismatch!")

ground state:  sigma_x = 0.7071,  sigma_p = 0.7071
product sigma_x * sigma_p = 0.5000
hbar/2                    = 0.5000
Minimum-uncertainty state confirmed.


## Part 2 · The Quantized Radiation Field

Here is the payoff for building all that machinery. A single mode of the electromagnetic field confined in an optical cavity behaves, mathematically, like *one more harmonic oscillator*. The same ladder operators return — we just rename them and reinterpret what they count.

### 2.1 Photons are oscillator quanta

For a single cavity mode of frequency $\omega_{\mathrm{cav}}$ we introduce photon ladder operators $\hat{b}$ and $\hat{b}^{\dagger}$ with the identical algebra,

$$
\hat{b}\,|n\rangle = \sqrt{n}\,|n-1\rangle, \qquad
\hat{b}^{\dagger}\,|n\rangle = \sqrt{n+1}\,|n+1\rangle, \qquad
[\hat{b}, \hat{b}^{\dagger}] = 1,
$$

and the cavity Hamiltonian

$$
\hat{H}_{\mathrm{cav}} = \hbar\omega_{\mathrm{cav}}\left(\hat{b}^{\dagger}\hat{b} + \tfrac{1}{2}\right).
$$

The only thing that has changed is *interpretation*. The eigenstate $|n\rangle$ is now a **photon number state** (or *Fock state*): it describes the mode containing exactly $n$ photons. $\hat{b}^{\dagger}$ **creates** a photon, $\hat{b}$ **destroys** one, and $\hat{b}^{\dagger}\hat{b}$ counts them. The ground state $|0\rangle$ is the electromagnetic **vacuum** — the mode with no photons — and its energy $\tfrac12\hbar\omega_{\mathrm{cav}}$ is the zero-point energy of the field.

We will keep the convention used in the accompanying chapter and midterm: **$\hat{a}, \hat{a}^{\dagger}$ act on the molecular (matter) oscillator; $\hat{b}, \hat{b}^{\dagger}$ act on the photon field.**

### 2.2 A word of intuition (and what we will *not* do here)

Why should light and molecular vibrations talk to each other at all? The starting point is the **minimal-coupling Hamiltonian**, in which a charged particle's momentum $\hat{p}$ is replaced by $\hat{p} - z\hat{A}$, where $z$ is the charge and $\hat{A}$ is the vector potential of the field. In words: *the field imparts momentum to the charges.* Expanding $(\hat{p} - z\hat{A})^2$ produces a term $\propto \hat{p}\cdot\hat{A}$ that directly couples the molecular momentum to the photonic vector potential — the seed of all light–matter interaction.

That expansion, and the unitary transformations that reshape it into the "dipole gauge" and Pauli–Fierz forms, are developed carefully in the chapter. **We will not reproduce that machinery here.** Our narrower goal is to understand the *building blocks* those Hamiltonians are made of — the field operators $\hat{A}$ and $\hat{E}$, their expectation values, and their fluctuations — so that the coupled Hamiltonians read as assemblies of familiar parts.

### 2.3 The dictionary: position $\leftrightarrow$ vector potential, momentum $\leftrightarrow$ electric field

The single most useful thing to carry from Part 1 into cavity QED is a **dictionary**. Just as the matter position and momentum were the sum and difference of matter ladder operators, the two fundamental field observables are the sum and difference of photon ladder operators:

$$
\hat{A} = A_0\left(\hat{b}^{\dagger} + \hat{b}\right), \qquad
\hat{E} = i\,E_0\left(\hat{b}^{\dagger} - \hat{b}\right),
$$

with the zero-point amplitudes

$$
A_0 = \sqrt{\frac{\hbar}{2\omega_{\mathrm{cav}}\epsilon_0 V}}, \qquad
E_0 = \sqrt{\frac{\hbar\omega_{\mathrm{cav}}}{2\epsilon_0 V}},
$$

where $V$ is the cavity mode volume and $\epsilon_0$ the vacuum permittivity. (We take all vectors along a single polarization direction $\hat{e}$, so dot products become ordinary products, exactly as in the chapter.)

The parallel is exact:

| matter oscillator | photon field | structure |
|---|---|---|
| position $\hat{x} \propto (\hat{a}^{\dagger} + \hat{a})$ | vector potential $\hat{A} \propto (\hat{b}^{\dagger} + \hat{b})$ | **sum** — real, symmetric |
| momentum $\hat{p} \propto i(\hat{a}^{\dagger} - \hat{a})$ | electric field $\hat{E} \propto i(\hat{b}^{\dagger} - \hat{b})$ | **difference** — imaginary, antisymmetric |

So $\hat{A}$ *is* the field's "position-like" coordinate and $\hat{E}$ *is* its "momentum-like" coordinate. Every result we proved for $\hat{x}$ and $\hat{p}$ now has an immediate photonic counterpart — including the minimum-uncertainty relation, which becomes the statement that the vacuum has irreducible field fluctuations.

In [12]:
# Photon operators are literally the same matrices as the matter ones.
# We reuse annihilation()/creation() and just call the quanta "photons".

def vector_potential(N, A0=1.0):
    """A = A0 (b^dag + b)  -- the field's 'position-like' operator."""
    return A0 * (creation(N) + annihilation(N))

def electric_field(N, E0=1.0):
    """E = i E0 (b^dag - b) -- the field's 'momentum-like' operator."""
    return 1j * E0 * (creation(N) - annihilation(N))

# Set A0 = E0 = 1 to expose the operator structure (the prefactors just rescale).
N = 4
A = vector_potential(N)
E = electric_field(N)

print("Vector potential A  ~ (b^dag + b)   [compare to x]:")
print(A.real)
print("\nElectric field E    ~ i(b^dag - b)  [compare to p]:")
print(E)

Vector potential A  ~ (b^dag + b)   [compare to x]:
[[0.    1.    0.    0.   ]
 [1.    0.    1.414 0.   ]
 [0.    1.414 0.    1.732]
 [0.    0.    1.732 0.   ]]

Electric field E    ~ i(b^dag - b)  [compare to p]:
[[ 0.+0.j    -0.-1.j     0.+0.j     0.+0.j   ]
 [ 0.+1.j     0.+0.j    -0.-1.414j  0.+0.j   ]
 [ 0.+0.j     0.+1.414j  0.+0.j    -0.-1.732j]
 [ 0.+0.j     0.+0.j     0.+1.732j  0.+0.j   ]]


Set these side by side with the $\hat{x}$ and $\hat{p}$ matrices from §1.5 — they are identical up to the constants $A_0$ and $E_0$. The dictionary is not an analogy in the loose sense; it is the *same matrices* wearing different physical labels.

### 2.4 Expectation values in a photon number state

What is the average electric field in a state with a definite number of photons? Using the dictionary and the ladder rules,

$$
\langle n|\hat{E}|n\rangle = i E_0\left(\langle n|\hat{b}^{\dagger}|n\rangle - \langle n|\hat{b}|n\rangle\right)
= i E_0\left(\sqrt{n+1}\,\langle n|n+1\rangle - \sqrt{n}\,\langle n|n-1\rangle\right) = 0,
$$

because $\hat{b}^{\dagger}$ and $\hat{b}$ map $|n\rangle$ onto *orthogonal* neighbors $|n\pm1\rangle$, which have zero overlap with $|n\rangle$. The same holds for $\hat{A}$. Let us confirm across several photon numbers.

In [13]:
N = 6   # padded so the states we print are away from the truncation ceiling
A = vector_potential(N)
E = electric_field(N)

print(f"{'n photons':>10} | {'<n|E|n>':>12} | {'<n|A|n>':>12}")
print("-" * 42)
for n in range(N - 1):   # skip the very top state (truncation-corrupted)
    print(f"{n:>10} | {expval(E, n, N).real:>12.3f} | {expval(A, n, N).real:>12.3f}")

 n photons |      <n|E|n> |      <n|A|n>
------------------------------------------
         0 |        0.000 |        0.000
         1 |        0.000 |        0.000
         2 |        0.000 |        0.000
         3 |        0.000 |        0.000
         4 |        0.000 |        0.000


Every average is zero. **This does not mean the field vanishes.** An expectation value is the average over many identical measurements; $\langle n|\hat{E}|n\rangle = 0$ says the field is equally likely to be measured pointing one way or the other, so it averages out. Any *single* measurement can still return a large value. To see that the field is emphatically *not* dead, we look at its fluctuations.

### 2.5 Vacuum fluctuations: $\langle \hat{E}^2\rangle \neq 0$

The spread of the field is governed by $\langle \hat{E}^2\rangle$, which is *not* zero even in the vacuum $|0\rangle$. This is the field's version of the oscillator's zero-point motion, and it is the origin of real, measurable effects (spontaneous emission, the Lamb shift, the Casimir force). By the dictionary it mirrors $\langle \hat{p}^2\rangle$ exactly:

$$
\langle n|\hat{E}^2|n\rangle = E_0^2\,(2n+1), \qquad \sigma_E = E_0\sqrt{2n+1}.
$$

The $n=0$ value, $\sigma_E = E_0 \neq 0$, is the **vacuum field fluctuation**.

In [14]:
N = 8
E = electric_field(N)   # E0 = 1
E2 = E @ E

print(f"{'n':>3} | {'<E^2>':>8} {'2n+1':>6} | {'sigma_E':>8}")
print("-" * 34)
for n in range(N - 2):   # stay well below the ceiling: E^2 reaches |n+2>
    e2 = expval(E2, n, N).real
    print(f"{n:>3} | {e2:>8.3f} {2*n+1:>6} | {np.sqrt(e2):>8.3f}")

print("\nGround-state (vacuum) field fluctuation sigma_E =",
      round(np.sqrt(expval(E2, 0, N).real), 4), " -> nonzero!")

  n |    <E^2>   2n+1 |  sigma_E
----------------------------------
  0 |    1.000      1 |    1.000
  1 |    3.000      3 |    1.732
  2 |    5.000      5 |    2.236
  3 |    7.000      7 |    2.646
  4 |    9.000      9 |    3.000
  5 |   11.000     11 |    3.317

Ground-state (vacuum) field fluctuation sigma_E = 1.0  -> nonzero!


The vacuum is not empty in the classical sense: the mode carries irreducible electric-field jitter even with zero photons. This is the same mathematics as the molecule's zero-point vibration $\sigma_x = \sqrt{\hbar/2m\omega}$ — the dictionary at work once more.

### 2.6 Putting matter and light together: the composite space

To describe a molecule *and* a cavity mode at once, we form the **tensor product** (Kronecker product) of the two spaces. A basis state is

$$
|n^{\mathrm{M}}\rangle \otimes |m^{\mathrm{P}}\rangle,
$$

the matter oscillator in state $n$ and the photon field in state $m$. If we keep $N_{\mathrm M}$ matter states and $N_{\mathrm P}$ photon states, the joint space has dimension $N_{\mathrm M}\times N_{\mathrm P}$.

Operators are lifted into this space with identities on the "other" subsystem:

- a purely matter operator becomes $\hat{O}_{\mathrm M}\otimes \mathbb{1}_{\mathrm P}$,
- a purely photonic operator becomes $\mathbb{1}_{\mathrm M}\otimes \hat{O}_{\mathrm P}$,
- and a genuine light–matter coupling is a product of a matter operator with a photon operator, e.g. $\hat{x}\otimes\hat{E}$.

In NumPy the tensor product is `np.kron`. This is precisely the construction used to build coupled light–matter Hamiltonians as matrices — the topic of the chapter and of the take-home problems. Here we build the pieces and check one property that always matters: **Hermiticity** (a valid observable must equal its own conjugate transpose).

In [15]:
Nm, Np = 2, 2          # keep 2 matter and 2 photon states -> 4-dim joint space

# subsystem building blocks
Im = np.eye(Nm)
Ip = np.eye(Np)
x_m = position(Nm)                 # matter position
b   = annihilation(Np)
bd  = creation(Np)
E_p = 1j * (bd - b)                # photon electric field (E0 = 1)
Ncav = bd @ b                      # photon number operator

# lift each piece into the composite (matter (x) photon) space
X_full    = np.kron(x_m,  Ip)      # matter operator, identity on photons
E_full    = np.kron(Im,   E_p)     # photon operator, identity on matter
Hcav_full = np.kron(Im,   Ncav)    # photon energy (hbar = omega_cav = 1)

# a schematic dipole-like coupling: (matter position) x (photon field)
#   -- illustrative only; the actual coupled Hamiltonians are built in the chapter
g = 0.1
H_coupling = g * np.kron(x_m, E_p)

print("Composite basis order: |0M,0P>, |0M,1P>, |1M,0P>, |1M,1P>")
print("\nCoupling block g * (x (x) E), shape", H_coupling.shape, ":")
print(H_coupling)

# Hermiticity test
def is_hermitian(M, tol=1e-12):
    return np.allclose(M, M.conj().T, atol=tol)

print("\nIs the coupling Hermitian? ", is_hermitian(H_coupling))
print("Is H_cav Hermitian?        ", is_hermitian(Hcav_full))

Composite basis order: |0M,0P>, |0M,1P>, |1M,0P>, |1M,1P>

Coupling block g * (x (x) E), shape (4, 4) :
[[0.+0.j    0.+0.j    0.+0.j    0.-0.071j]
 [0.+0.j    0.+0.j    0.+0.071j 0.+0.j   ]
 [0.+0.j    0.-0.071j 0.+0.j    0.+0.j   ]
 [0.+0.071j 0.+0.j    0.+0.j    0.+0.j   ]]

Is the coupling Hermitian?  True
Is H_cav Hermitian?         True


The coupling is Hermitian because both factors are: $\hat{x}$ is real-symmetric, and $\hat{E} = i(\hat{b}^{\dagger}-\hat{b})$ is Hermitian precisely because the factor of $i$ compensates the antisymmetry of $\hat{b}^{\dagger}-\hat{b}$. (Check: $[\,i(\hat{b}^{\dagger}-\hat{b})\,]^{\dagger} = -i(\hat{b}-\hat{b}^{\dagger}) = i(\hat{b}^{\dagger}-\hat{b})$.) This is the same reason $\hat{p}$ is Hermitian, and it is why the electric-field coupling carries an explicit $i$ while the vector-potential coupling does not. When you assemble a coupled Hamiltonian, Hermiticity is a fast, cheap sanity check that the operators went in correctly.

### Try it yourself

Short exercises to cement the formalism. None require anything beyond the helpers already defined.

1. **Excited-state uncertainty.** Adapt §1.9 to compute $\sigma_x\sigma_p$ for $|1\rangle$ and $|2\rangle$. Confirm the product grows as $(2n+1)\,\hbar/2$, so only the ground state saturates the Heisenberg bound. (Remember to pad $N$.)
2. **Photon-field uncertainty.** Show numerically that $\sigma_A\,\sigma_E$ in the vacuum equals $A_0 E_0$, the photonic minimum-uncertainty relation — the field analog of Exercise 1.
3. **Truncation hunt.** For $N = 3$, print $\langle n|\hat{x}^2|n\rangle$ for all $n$ and identify which entries are wrong. Then increase $N$ until the $n=2$ value is correct, and check it matches the padding rule of §1.6.
4. **Number-state field, visualized.** For a fixed $n$, plot the "field quadrature distribution" by histogramming... or, more simply, tabulate $\langle n|\hat{E}^k|n\rangle$ for $k = 1, 2, 3, 4$ and note which vanish by symmetry.
5. **Coupling matrix.** Build the composite operator $\hat{a}^{\dagger}\hat{a}\otimes\mathbb{1} + \mathbb{1}\otimes\hat{b}^{\dagger}\hat{b} + g(\hat{a}^{\dagger}+\hat{a})\otimes(\hat{b}^{\dagger}+\hat{b})$ for $N_{\mathrm M}=N_{\mathrm P}=2$, confirm it is Hermitian, and diagonalize it to find the coupled energy levels as a function of $g$.

### That's a wrap!

In this notebook we:

- rebuilt the harmonic oscillator in the language of **ladder operators**, and expressed $\hat{x}$, $\hat{p}$, $\hat{N}$, and $\hat{H}$ through $\hat{a}$ and $\hat{a}^{\dagger}$;
- represented those operators as **finite matrices** in a truncated Fock basis and watched their matrix elements and commutators directly;
- saw that **basis truncation** necessarily breaks $[\hat{a},\hat{a}^{\dagger}]=\mathbb{1}$ — because a finite commutator is traceless — and that the damage is confined to the top of the ladder, giving a concrete rule for how large to make the basis;
- reused the entire construction for the **quantized cavity field**, building the dictionary $\hat{x}\leftrightarrow\hat{A}$ and $\hat{p}\leftrightarrow\hat{E}$;
- computed field **expectation values** ($\langle n|\hat{E}|n\rangle = 0$) and **fluctuations** ($\sigma_E \neq 0$ even in vacuum), and interpreted the latter as vacuum field fluctuations;
- assembled the **composite matter $\otimes$ photon space** with Kronecker products and used Hermiticity as a check.

With these tools, the coupled light–matter Hamiltonians of the next chapter are just familiar blocks stacked together with `np.kron`.

### Read these next

- The accompanying chapter, which develops the minimal-coupling, dipole-gauge, and Pauli–Fierz Hamiltonians and their matrix representations via unitary transformations.
- *Introductory Quantum Optics*, C. Gerry and P. Knight (Cambridge University Press, 2004) — a clear undergraduate/early-graduate treatment of field quantization and number states.
- *Quantum Optics: An Introduction*, M. Fox (Oxford University Press, 2006) — an accessible complement with strong physical intuition.